# Exp04 λ grid for BEREL 3.0

**Runtime → Run all.** A GPU runtime is recommended but not required — see *What it costs* below.

## Why this notebook exists

`run_cross_data_model_comparison.py` runs the λ grid **once per run directory**, in the `[PREP]` block *before* the model loop starts:

```python
ensure_exp04_loss_weights_calibrated(COMPARISON_DIR, ...)   # line ~2494
...
for model_info in models:
    _set_model_env(model_info["model_id"])                  # line ~2781
```

The grid resolves its encoder through `configure_model_environment()`, and at that point `THESIS_MODEL_NAME` is still unset — so it falls back to `DEFAULT_MODEL_ID = "dicta-il/dictabert"`. The λ pair in `journal_lambda_grid` is therefore **DictaBERT's**, and BEREL's Exp04 and OOF runs simply inherit it.

This notebook runs the **identical** grid with `THESIS_MODEL_NAME` pinned to `dicta-il/BEREL_3.0`: same `{1,5,10}²` values, same 70/30 validation split, same split seed, same fair training defaults, same selection metric (validation pipeline span F1 in predicted mode after the threshold sweep). Only the encoder changes.

## What it costs

Nine Exp04 trainings. The corpus is 150 sentences (~105 for training), so each one is small: the 35 Exp04 runs on this corpus in `outputs/exp04/` (Aug 2026, **CPU-only** laptop, `torch 2.11.0+cpu`) took a median of **5.3 minutes** each, min 4.9, max 9.6. Early stopping (patience 3, `min_delta` 1e-3) ended most of them around epoch 4 of 10.

So expect roughly **45–60 minutes on CPU** and **20–40 minutes on a Colab T4**, plus a one-time BEREL download. The worst case is a λ cell where validation F1 keeps improving and training runs the full 10 epochs; a grid of nine of those would be around 2 hours on CPU.

A GPU is therefore a convenience here, not a requirement — the corpus is too small for the GPU to pay off the way it does on a full run. Note that Colab's **CPU** runtime also drops the Exp04 batch size to 4 with 4 accumulation steps (`core/runtime_env.py`), so it is somewhat slower per epoch than the laptop numbers above.

It is resumable either way: every finished cell is written to `exp04_lambda_grid_progress.json` on Drive, so after a disconnect you re-run cells 1–5 and then cell 6, and only the unfinished cells train.

## What it does *not* touch

Nothing from the completed training run is modified. The grid writes to its own `exp04_lambda_grid_berel/` folder with its own `THESIS_EXPERIMENTS_OUTPUT_DIR`, so the nine throwaway Exp04 workbooks never land in the main run's `experiment_outputs/exp04/` and cannot be mistaken for real results by the error-analysis consolidation.

## 1 — Mount Google Drive

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running on Colab — skipping the Drive mount.")

## 2 — Paths

`OUTPUT_DIR` is the training run's directory — the grid reads `training_seeds.json` and the DictaBERT λ selection from it, and writes its own results into a subfolder of it.

In [ ]:
import os
from pathlib import Path

WORK_ROOT = Path("/content/drive/MyDrive/thesis_project")
OUTPUT_DIR = WORK_ROOT / "5folds_run"

# Code on the Colab disk, not Drive: git checkouts on a Drive mount often break.
REPO_DIR = Path("/content/thesis")
REPO_URL = "https://github.com/menachemsperka/thesis.git"

NER_DATASET_PKL = REPO_DIR / "data" / "ner_dataset_150_seed42.pkl"

# Everything this notebook produces, isolated from the main run's artifacts.
GRID_ROOT = OUTPUT_DIR / "exp04_lambda_grid_berel"
CACHE_DIR = GRID_ROOT / "cache"                            # progress + selection + grid xlsx
GRID_EXPERIMENT_OUTPUTS = GRID_ROOT / "experiment_outputs"  # the 9 throwaway Exp04 workbooks
LOG_PATH = GRID_ROOT / "lambda_grid_berel.log"

# The DictaBERT selection the runner already produced, for side-by-side comparison.
DICTABERT_SELECTION = OUTPUT_DIR / "exp04_lambda_grid_cache" / "exp04_lambda_selection.json"

for d in (GRID_ROOT, CACHE_DIR, GRID_EXPERIMENT_OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("GRID_ROOT: ", GRID_ROOT)
print("REPO_DIR:  ", REPO_DIR)

## 3 — Clone or update the repository

The repo is public, so no token. The 150-sentence pickle ships under `data/`, so nothing to upload.

In [ ]:
import shutil
import subprocess


def _git_head_ok(repo):
    if not (repo / ".git").is_dir():
        return False
    return (
        subprocess.run(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True
        ).returncode
        == 0
    )


# Step out of REPO_DIR first: on a re-run the cwd is inside it, and rmtree would
# leave the process on a deleted directory so the clone that follows would fail.
os.chdir(REPO_DIR.parent)

if _git_head_ok(REPO_DIR):
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    print("Updated existing clone.")
else:
    if REPO_DIR.is_dir():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", "main", "--single-branch",
            REPO_URL, str(REPO_DIR),
        ],
        check=True,
    )
    print("Cloned repository.")

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

for required in (
    REPO_DIR / "experiments" / "experiment_04_loss_weight_grid.py",
    REPO_DIR / "core" / "auc_cascaded_pipeline.py",
    NER_DATASET_PKL,
):
    if not required.is_file():
        raise FileNotFoundError(f"Missing {required} in the clone.")
print("Grid script, pipeline and corpus present.")

## 4 — Install dependencies

In [ ]:
import sys

os.chdir(REPO_DIR)
# Build tools first: seqeval is a legacy setup.py package and fails without them.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy", "openpyxl"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "seqeval", "--no-build-isolation"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
)

os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
print("Dependencies OK.")

## 5 — Grid configuration and preflight

**The split seed is not a free choice.** The runner calibrates with `THESIS_SPLIT_SEED = seeds[0]` from `training_seeds.json`, and the DictaBERT selection file records which seed it actually used. This cell prefers that recorded seed, so the two grids differ *only* in the encoder and the comparison between them is clean.

`STREAM_TRAINING_LOGS = True` sets `THESIS_DEBUG=1` so each Exp04 training streams into the output and the log file. Without it `experiment_04_auc_cascaded_pipeline.run()` captures the child's output and a ten-minute training looks like a hang. The trade-off: on failure the exception message says `No subprocess output captured`, because the traceback went to the log instead.

In [ ]:
import json

MODEL_KEY = "berel"
LAMBDA_VALUES = "1,5,10"       # {1,5,10}^2 = 9 cells, the same grid DictaBERT used
FORCE_REBUILD = False          # True retrains every cell, discarding cached ones
STREAM_TRAINING_LOGS = True

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Reuse the runner's own registry and local-first path resolution rather than
# hard-coding the model id, so this notebook cannot drift from the comparison run.
from run_cross_data_model_comparison import MODEL_REGISTRY, _resolve_local_model_path

MODEL_ID = MODEL_REGISTRY[MODEL_KEY]["model_id"]
MODEL_PATH = _resolve_local_model_path(MODEL_ID)

dictabert_sel = None
if DICTABERT_SELECTION.is_file():
    dictabert_sel = json.loads(DICTABERT_SELECTION.read_text(encoding="utf-8"))

if dictabert_sel and dictabert_sel.get("split_seed") is not None:
    SPLIT_SEED = int(dictabert_sel["split_seed"])
    seed_source = f"DictaBERT selection ({DICTABERT_SELECTION.name})"
else:
    seeds_path = OUTPUT_DIR / "training_seeds.json"
    if seeds_path.is_file():
        stored = [int(s) for s in json.loads(seeds_path.read_text(encoding="utf-8")).get("seeds", [])]
        SPLIT_SEED = stored[0] if stored else 42
        seed_source = f"training_seeds.json seeds[0] ({len(stored)} seeds on disk)"
    else:
        SPLIT_SEED = 42
        seed_source = "fallback default (no seed file found)"

if not NER_DATASET_PKL.is_file():
    raise FileNotFoundError(f"Missing corpus {NER_DATASET_PKL} — re-run cell 3.")

try:
    import torch

    cuda = torch.cuda.is_available()
except ModuleNotFoundError:
    cuda = False
print("CUDA available:", cuda)
print(
    "Estimated total: ~20-40 min on a T4"
    if cuda
    else "Estimated total: ~45-90 min on CPU (workable; the corpus is only 150 sentences)"
)

print(f"\nmodel          : {MODEL_KEY} -> {MODEL_ID}")
print(f"resolved to    : {MODEL_PATH}" + ("  (local)" if Path(MODEL_PATH).exists() else "  (download from HF)"))
print(f"lambda values  : {LAMBDA_VALUES}  ->  {len(LAMBDA_VALUES.split(','))**2} training runs")
print(f"split seed     : {SPLIT_SEED}   [from {seed_source}]")
print(f"corpus         : {NER_DATASET_PKL}")
print(f"cache dir      : {CACHE_DIR}")

if dictabert_sel:
    print(
        "\nDictaBERT reference: "
        f"lambda_bio={dictabert_sel.get('selected_lambda_bio')}, "
        f"lambda_type={dictabert_sel.get('selected_lambda_type')}, "
        f"validation F1={dictabert_sel.get('selected_validation_pipeline_span_f1')}"
    )
else:
    print(
        f"\nNo DictaBERT selection at {DICTABERT_SELECTION} — the BEREL grid still runs, "
        "but cell 7 will have nothing to compare against."
    )

progress_file = CACHE_DIR / "exp04_lambda_grid_progress.json"
if progress_file.is_file():
    done = json.loads(progress_file.read_text(encoding="utf-8")).get("completed_pairs", {})
    print(f"\nResumable: {len(done)} of 9 cells already trained -> {sorted(done)}")
else:
    print("\nNo cached progress — this will be a full 9-cell run.")

## 6 — Run the grid

`run_grid()` is a library function with no CLI of its own, and only its `cache_dir` form writes per-cell progress — which is what makes a dropped Colab session resumable. So this runs it in a child process via `python -c`, with paths handed over through environment variables, and streams the output to `lambda_grid_berel.log` on Drive.

`THESIS_EXPERIMENTS_OUTPUT_DIR` points at this notebook's own folder, which is the one line that keeps the nine throwaway Exp04 workbooks out of the main run's `experiment_outputs/exp04/` — where they would otherwise be the newest files in the directory and could be picked up as if they were real results.

In [ ]:
runner_code = """
import json
import os
import sys
from pathlib import Path

repo = Path(os.environ["THESIS_GRID_REPO_DIR"])
cache = Path(os.environ["THESIS_GRID_CACHE_DIR"])
for entry in (str(repo / "experiments"), str(repo)):
    if entry not in sys.path:
        sys.path.insert(0, entry)

from core.runtime_env import describe_runtime
from core.training_defaults import apply_fair_comparison_training_defaults

# The runner applies these before calibrating, so the grid must too or the nine
# trainings would not match the Exp04 runs the selected lambda is meant for.
fair = apply_fair_comparison_training_defaults()
print(describe_runtime(), flush=True)
print("model        :", os.environ.get("THESIS_MODEL_NAME"), flush=True)
print("split seed   :", os.environ.get("THESIS_SPLIT_SEED"), flush=True)
print("exp04 epochs :", fair.get("THESIS_EXP04_EPOCHS"), flush=True)
print("grid values  :", os.environ.get("THESIS_EXP04_LAMBDA_GRID_VALUES"), flush=True)

from experiment_04_loss_weight_grid import run_grid

result = run_grid(
    cache_dir=cache,
    resume=os.environ.get("THESIS_GRID_RESUME") == "1",
    force_rebuild=os.environ.get("THESIS_GRID_FORCE") == "1",
)

# Same filename and key names exp04_loss_calibration.py writes, so
# apply_selection_to_env() can read this file unchanged.
selection = {
    "selected_lambda_bio": result.get("selected_lambda_bio"),
    "selected_lambda_type": result.get("selected_lambda_type"),
    "selected_validation_pipeline_span_f1": result.get("selected_validation_pipeline_span_f1"),
    "baseline_10_5_f1": result.get("baseline_10_5_f1"),
    "grid_excel": result.get("grid_excel"),
    "selection_metric": (result.get("training_parameters") or {}).get("selection_metric"),
    "model": result.get("model"),
    "model_key": os.environ.get("THESIS_GRID_MODEL_KEY"),
    "split_seed": (result.get("training_parameters") or {}).get("split_seed"),
}
out = cache / "exp04_lambda_selection.json"
out.write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding="utf-8")
print("Selection written to", out, flush=True)
print("status:", result.get("status"), flush=True)
"""

env = os.environ.copy()
env.update({
    "PYTHONUNBUFFERED": "1",
    "THESIS_RUN_ENV": "colab",
    "WANDB_DISABLED": "true",
    "THESIS_NER_CSV": str(NER_DATASET_PKL),
    "THESIS_MODEL_NAME": MODEL_PATH,
    "THESIS_SPLIT_SEED": str(SPLIT_SEED),
    "THESIS_EXP04_LAMBDA_GRID_VALUES": LAMBDA_VALUES,
    "THESIS_EXPERIMENTS_OUTPUT_DIR": str(GRID_EXPERIMENT_OUTPUTS),
    "THESIS_DELETE_MODELS_AFTER_TRAIN": "1",
    "THESIS_GRID_REPO_DIR": str(REPO_DIR),
    "THESIS_GRID_CACHE_DIR": str(CACHE_DIR),
    "THESIS_GRID_MODEL_KEY": MODEL_KEY,
    "THESIS_GRID_RESUME": "1",
    "THESIS_GRID_FORCE": "1" if FORCE_REBUILD else "0",
    "THESIS_DEBUG": "1" if STREAM_TRAINING_LOGS else "0",
})

# run_grid() sets these per cell; an inherited value would silently override cell 1 of 9.
for stale in ("THESIS_EXP04_LAMBDA_BIO", "THESIS_EXP04_LAMBDA_TYPE"):
    env.pop(stale, None)

if Path(MODEL_PATH).exists():
    env.update({"TRANSFORMERS_OFFLINE": "1", "HF_HUB_OFFLINE": "1", "THESIS_MODEL_LOCAL_ONLY": "1"})
else:
    for online in ("TRANSFORMERS_OFFLINE", "HF_HUB_OFFLINE", "THESIS_MODEL_LOCAL_ONLY"):
        env.pop(online, None)

n_cells = len(LAMBDA_VALUES.split(",")) ** 2
print(f"Training {n_cells} Exp04 configurations for {MODEL_ID}.")
print(f"Reference cost: ~5 min per cell on CPU, so ~{5 * n_cells} min for a cold start.")
print(f"Log: {LOG_PATH}\n")

# Popen + line iteration, not subprocess.run: run() inherits the raw fd and Colab
# then shows nothing until the process exits, which makes hours of training look
# like a hang. Mirror to Drive so a dropped connection does not lose the output.
with open(LOG_PATH, "a", encoding="utf-8") as logf, subprocess.Popen(
    [sys.executable, "-c", runner_code],
    cwd=str(REPO_DIR),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(
        f"Grid exited with code {proc.returncode}. Completed cells are cached in "
        f"{CACHE_DIR / 'exp04_lambda_grid_progress.json'} — fix the error and re-run "
        f"this cell to continue from where it stopped. Full output: {LOG_PATH}"
    )

print("\nGrid complete.")

## 7 — The λ surface and the selected pair

`heatmap_f1` is the 3×3 validation F1 surface (rows `λ_bio`, columns `λ_type`). Ties in `_select_best()` break toward the **smaller** λ, so a flat surface returns the least aggressive reweighting rather than an arbitrary corner.

If BEREL's selection matches DictaBERT's, or sits on a flat region around it, the existing results stand as they are and the grid becomes the ablation evidence that the choice was not arbitrary.

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

grid_xlsx = sorted(CACHE_DIR.glob("loss_weight_grid_*.xlsx"))
if not grid_xlsx:
    raise FileNotFoundError(f"No loss_weight_grid_*.xlsx under {CACHE_DIR} — did cell 6 finish?")
latest_xlsx = grid_xlsx[-1]
print("Reading", latest_xlsx.name, "\n")

sheets = pd.read_excel(latest_xlsx, sheet_name=None)

print("=== heatmap_f1: validation pipeline span F1 (rows lambda_bio, cols lambda_type) ===")
print(sheets["heatmap_f1"].round(4).to_string())

print("\n=== grid_runs: all 9 cells ===")
runs = sheets["grid_runs"].sort_values("validation_pipeline_span_f1", ascending=False)
print(
    runs[["lambda_bio", "lambda_type", "validation_pipeline_span_f1", "split_seed", "status"]]
    .to_string(index=False)
)

berel_sel = json.loads((CACHE_DIR / "exp04_lambda_selection.json").read_text(encoding="utf-8"))

rows = [{"encoder": "BEREL 3.0", **{k: berel_sel.get(k) for k in (
    "selected_lambda_bio", "selected_lambda_type",
    "selected_validation_pipeline_span_f1", "baseline_10_5_f1", "split_seed",
)}}]
if dictabert_sel:
    rows.append({"encoder": "DictaBERT", **{k: dictabert_sel.get(k) for k in (
        "selected_lambda_bio", "selected_lambda_type",
        "selected_validation_pipeline_span_f1", "baseline_10_5_f1", "split_seed",
    )}})

print("\n=== selection, BEREL vs DictaBERT ===")
print(pd.DataFrame(rows).to_string(index=False))
print("\nselection metric:", berel_sel.get("selection_metric"))

bio = berel_sel.get("selected_lambda_bio")
typ = berel_sel.get("selected_lambda_type")
if dictabert_sel and (bio, typ) == (
    dictabert_sel.get("selected_lambda_bio"),
    dictabert_sel.get("selected_lambda_type"),
):
    print(
        f"\nBEREL selects the same pair as DictaBERT ({bio}, {typ}) — the completed runs "
        "already used it, so no results change and this grid is pure ablation evidence."
    )
else:
    print(f"\nBEREL selects ({bio}, {typ}) — different from the pair its runs were trained with.")
    print("To retrain BEREL under its own weights, see the note in section 8.")

## 8 — Where everything landed, and how to use the result

**The runner will not pick this up on its own.** `calibration_cache_dir()` is hard-coded to `<output_dir>/exp04_lambda_grid_cache` and holds one λ pair for *all* models, so a BEREL-specific selection has to be applied by hand. For a BEREL-only run under its own weights:

```python
env["THESIS_EXP04_LAMBDA_BIO"]  = str(bio)   # from cell 7
env["THESIS_EXP04_LAMBDA_TYPE"] = str(typ)
# ... --models berel --skip-exp04-loss-calibration --output-dir <a NEW directory>
```

Use a **new** `--output-dir`. Checkpoint rows carry no λ, so resuming the existing run with different weights would mix two training profiles in one table — exactly the confound the unified-protocol table exists to remove.

In [ ]:
for label, path in [
    ("BEREL selection (the answer)", CACHE_DIR / "exp04_lambda_selection.json"),
    ("BEREL grid workbook", latest_xlsx),
    ("resume progress", CACHE_DIR / "exp04_lambda_grid_progress.json"),
    ("run log", LOG_PATH),
    ("DictaBERT selection (runner's, untouched)", DICTABERT_SELECTION),
    ("throwaway Exp04 workbooks", GRID_EXPERIMENT_OUTPUTS),
]:
    if path.is_dir():
        print(f"{label}:\n  {path}  ({len(list(path.rglob('*.xlsx')))} workbooks)")
    elif path.is_file():
        print(f"{label}:\n  {path}  ({path.stat().st_size:,} bytes)")
    else:
        print(f"{label}:\n  {path}  (missing)")